In [ ]:
import json
import matplotlib.pyplot as plt

# =========================
# Load JSON
# =========================
JSON_PATH = "../data/CDG-FCO/RESULTS/K_COMPARISON.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    raw = json.load(f)

data = {int(k): v for k, v in raw.items()}
Ks = sorted(data.keys())
PHASES = ["take_off", "cruising", "landing"]

def pareto_front_vs_k(Ks, errors):
    """
    Pareto over (K, error) with both to be minimized.
    Since K is 1D and increasing, the Pareto set is the sequence of
    'record lows' in error as K increases.
    Returns list of (k, e) on the frontier.
    """
    frontier = []
    best = float("inf")
    for k, e in zip(Ks, errors):
        if e < 0.945*best:
            frontier.append((k, e))
            best = e
    return frontier

plt.figure(figsize=(10, 6))

for phase in PHASES:
    errors = [data[k][phase]["error"] for k in Ks]

    # all points curve
    plt.plot(Ks, errors, marker="o", alpha=0.35, label=f"{phase} (all)")

    # pareto/frontier points
    front = pareto_front_vs_k(Ks, errors)
    fK = [p[0] for p in front]
    fE = [p[1] for p in front]

    plt.plot(fK, fE, marker="o", linewidth=2.5, label=f"{phase} Pareto")

    # label Pareto points
    for k, e in front:
        plt.annotate(f"K={k}", (k, e), textcoords="offset points", xytext=(6, 6), fontsize=9)

plt.xlabel("K (number of retrieved neighbors)")
plt.ylabel("Mean Haversine error (km)")
plt.title("Pareto Frontier — Mean Haversine Error vs K (per phase)")
plt.xticks(Ks)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()
